In [6]:
import sys
import subprocess

# List of required packages
packages = [
    "pandas",
    "numpy",
    "tensorflow",
    "scikit-learn",
    "joblib"
]

# Install each package
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
import joblib
import os


In [14]:
!{sys.executable} -m pip install matplotlib

You should consider upgrading via the 'D:\AB\Barclay\mlmodel\M2\failurepredict\Scripts\python.exe -m pip install --upgrade pip' command.


In [33]:
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import pickle
import joblib
import os

# Function to load and process log files
def load_logs(files):
    all_data = []
    
    for file in files:
        with open(file, 'r') as f:
            for line in f:
                try:
                    log_entry = json.loads(line.strip())
                    all_data.append(log_entry)
                except json.JSONDecodeError:
                    print(f"Skipping invalid JSON line in {file}")
    
    return all_data

# Load log data
log_files = ['api_logs.json', 'test_logs.json']
logs = load_logs(log_files)

# Convert logs to DataFrame
df = pd.DataFrame(logs)

# Feature engineering
# Extract features that might indicate system stress
df_features = pd.DataFrame()

# Response time is a key indicator of system stress
df_features['response_time_ms'] = df['response_time_ms']

# Extract hour from timestamp for time-based patterns
df_features['hour'] = pd.to_datetime(df['timestamp']).dt.hour

# Status code - convert to binary (success/error)
df_features['is_error'] = (df['status_code'] >= 400).astype(int)

# Request body size might impact performance
df_features['request_size'] = df.apply(
    lambda row: len(str(row['request_body'])) if isinstance(row['request_body'], dict) else 0, 
    axis=1
)

# Environment feature (encoded)
df_features['is_onprem'] = (df['environment'] == 'on-prem').astype(int)

# Define target variable - assuming high response times indicate an API about to crash
# Let's define "about to crash" as response times in the top 15%
threshold = df['response_time_ms'].quantile(0.85)
df_features['about_to_crash'] = (df['response_time_ms'] > threshold).astype(int)

# Remove response time from features as it's used for the target (to avoid data leakage)
X = df_features.drop(['response_time_ms', 'about_to_crash'], axis=1)
y = df_features['about_to_crash']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build neural network model
model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(8, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

# Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)
print(f"Test accuracy: {test_accuracy:.4f}")

# Save the model
model.save('h2.h5')

# Save the scaler
joblib.dump(scaler, 'scaler2.save')

print("Model and scaler saved successfully.")

# Function to make predictions with the model
def predict_api_crash(new_data):
    """
    Predict if API is about to crash based on new log data
    
    Parameters:
    new_data: A dictionary with log data fields
    
    Returns:
    Probability of API crash
    """
    # Create features from new data
    features = {
        'hour': pd.to_datetime(new_data.get('timestamp', pd.Timestamp.now())).hour,
        'is_error': 1 if new_data.get('status_code', 200) >= 400 else 0,
        'request_size': len(str(new_data.get('request_body', {}))),
        'is_onprem': 1 if new_data.get('environment', '') == 'on-prem' else 0
    }
    
    # Convert to DataFrame
    df_pred = pd.DataFrame([features])
    
    # Scale features
    scaled_features = scaler.transform(df_pred)
    
    # Make prediction
    crash_probability = model.predict(scaled_features)[0][0]
    
    return crash_probability

# Example usage
example_log = {
    "timestamp": "2025-04-26 18:22:14.934681",
    "status_code": 200,
    "request_body": {"name": "Test", "value": 42},
    "environment": "on-prem"
}

crash_probability = predict_api_crash(example_log)
print(f"Probability of API crash: {crash_probability:.4f}")
print(f"Prediction: {'API may crash soon' if crash_probability > 0.5 else 'API stable'}")

D:\AB\Barclay\mlmodel\M2\failurepredict\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.2802 - loss: 0.7383 - val_accuracy: 0.1290 - val_loss: 0.7324
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2910 - loss: 0.7261 - val_accuracy: 0.1290 - val_loss: 0.7123
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4745 - loss: 0.6959 - val_accuracy: 0.7097 - val_loss: 0.6958
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5389 - loss: 0.6851 - val_accuracy: 0.7097 - val_loss: 0.6815
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5944 - loss: 0.6757 - val_accuracy: 0.7097 - val_loss: 0.6691
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6187 - loss: 0.6609 - val_accuracy: 0.7097 - val_loss: 0.6577
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6149 - loss: 0.6517 - val_accuracy: 0.7097 - val_loss: 0.6467
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6723 - loss: 0.6459 - val_accuracy: 0.7097 - val_loss: 0.6357


Test accuracy: 0.8718
Model and scaler saved successfully.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
Probability of API crash: 0.1075
Prediction: API stable
